In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error,root_mean_squared_error, r2_score


In [2]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

# Isolate features (X) and target outputs (y)
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [7]:
# Train an unconstrained tree
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

# Generate predictions for both sets
train_pred = tree.predict(X_train)
test_pred = tree.predict(X_test)

# Calculate errors
train_rmse = mean_squared_error(y_train, train_pred) # or np.sqrt()
test_rmse = mean_squared_error(y_test, test_pred)

print(f"Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

Train RMSE: 0.0000 | Test RMSE: 0.4930


In [8]:
cv_scores = cross_val_score(
    tree, X_scaled, y,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# Convert negative scores back to positive RMSE
cv_rmse = -cv_scores.mean()
print(f"5-Fold Cross-Validated RMSE: {cv_rmse:.4f}")

5-Fold Cross-Validated RMSE: 0.8962


In [9]:
# Define hyperparameter options to evaluate
param_grid = {
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10]
}

# Setup the grid search
grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5
)

# Execute the search grid sweep
grid.fit(X_train, y_train)

print(f"Optimized Parameters Found: {grid.best_params_}")

Optimized Parameters Found: {'max_depth': 10, 'min_samples_split': 10}


In [11]:
best_tree = grid.best_estimator_
y_pred = best_tree.predict(X_test)

final_rmse = mean_squared_error(y_test, y_pred)
final_r2 = r2_score(y_test, y_pred)

print(f"Tuned Tree - RMSE: {final_rmse:.4f}, R2 Score: {final_r2:.4f}")

Tuned Tree - RMSE: 0.4166, R2 Score: 0.6821


In [13]:
results = {
    "Model": ["Linear Regression", "Ridge Regression", "Tuned Decision Tree"],
    "RMSE": [0.5559, 0.5558, final_rmse],
    "R2 Score": [0.5758, 0.5758, final_r2]
}

comparison_df = pd.DataFrame(results)
print(comparison_df)

                 Model     RMSE  R2 Score
0    Linear Regression  0.55590  0.575800
1     Ridge Regression  0.55580  0.575800
2  Tuned Decision Tree  0.41658  0.682099


In [14]:

with open("model_validation_comparison.md", "w") as f:
    f.write("# Task 3: Model Performance Validation & Tuning Summary\n\n")
    f.write(comparison_df.to_markdown(index=False))

print("Markdown validation summary table successfully written!")

Markdown validation summary table successfully written!
